<a href="https://colab.research.google.com/github/BakrAdli/My_AI_Journey/blob/main/smart_car_diagnostics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# V2.7 PRO : Type-Hinted, Memory-Optimized & Environment-Agnostic
import json
import os
from typing import Dict, List, Optional, Union, Any

# --- Universal Clear Screen Function ---
def clear_screen() -> None:
    """Clears the terminal/output regardless of the OS or environment."""
    try:
        from IPython.display import clear_output
        clear_output(wait=True)
    except ImportError:
        os.system('cls' if os.name == 'nt' else 'clear')

class Car:
    def __init__(
        self,
        brand: str,
        model: str,
        engine_type: str,
        mileage: int,
        last_oil_change: int = 0,
        last_tire_change: int = 0,
        trip_history: Optional[List[int]] = None
    ) -> None:
        self.brand = brand
        self.model = model
        self.engine_type = engine_type
        self.mileage = mileage
        self.last_oil_change = last_oil_change
        self.last_tire_change = last_tire_change
        self.trip_history = trip_history if trip_history is not None else []

    def add_trip(self, distance: int) -> bool:
        if distance <= 0:
            print("\n  [Error] Distance must be > 0. Cannot add negative/zero km.")
            return False

        self.mileage += distance
        self.trip_history.append(distance)

        # Memory Optimization: Keep only the last 50 trips to prevent JSON bloat
        if len(self.trip_history) > 50:
            self.trip_history = self.trip_history[-50:]

        print(f"\n  [System] Trip of {distance:,} km recorded! Odometer: {self.mileage:,} km")
        self.save_data(silent=True)
        return True

    def reset_service(self, service_type: str) -> None:
        """Unified method for resetting services efficiently."""
        if service_type == "oil":
            self.last_oil_change = self.mileage
            print(f"\n  [Service] Oil reset at {self.mileage:,} km!")
        elif service_type == "tires":
            self.last_tire_change = self.mileage
            print(f"\n  [Service] Tires reset at {self.mileage:,} km!")
        self.save_data(silent=True)

    def edit_data(self) -> None:
        print("\n[*] Edit Vehicle Details (Press Enter to keep current value): ")

        new_brand = input(f" Brand [{self.brand}] : ").strip()
        if new_brand: self.brand = new_brand

        new_model = input(f" Model [{self.model}] : ").strip()
        if new_model: self.model = new_model

        new_engine = input(f" Engine [{self.engine_type}] : ").strip()
        if new_engine: self.engine_type = new_engine

        while True:
            new_mile = input(f" Mileage [{self.mileage}] : ").strip()
            if not new_mile:
                break
            try:
                temp_mileage = int(new_mile)
                if temp_mileage < self.mileage:
                    print(f"   [Error] Fraud Alert! New mileage ({temp_mileage:,}) < current ({self.mileage:,})!")
                    continue
                self.mileage = temp_mileage
                break
            except ValueError:
                print("   [Error] Invalid input. Positive integers only.")

        self.save_data(silent=True)
        print("  [System] Data updated successfully!")

    def get_oil_status(self) -> str:
        km_driven = self.mileage - self.last_oil_change
        if km_driven >= 10000:
            return f"CHANGE_REQUIRED ({km_driven:,} km driven)"
        return f"OK ({10000 - km_driven:,} km left)"

    def get_tires_status(self) -> str:
        km_driven = self.mileage - self.last_tire_change
        if km_driven >= 50000:
            return f"CHECK_TREAD ({km_driven:,} km driven)"
        return f"OK ({50000 - km_driven:,} km left)"

    def get_engine_analysis(self) -> Dict[str, str]:
        engine_lower = self.engine_type.lower()
        if any(kw in engine_lower for kw in ["diesel", "crdi"]):
            return {"type": "DIESEL/CRDi", "tip": "Check Glow Plugs & EGR Valve"}
        elif "hybrid" in engine_lower:
            return {"type": "HYBRID", "tip": "Schedule battery health diagnostic"}
        return {"type": "GASOLINE", "tip": "Check spark plugs & fuel injectors"}

    def get_llm_context(self) -> str:
        context = {
            "vehicle": f"{self.brand} {self.model}",
            "mileage": self.mileage,
            "history": {
                "oil_change_at": self.last_oil_change,
                "tire_change_at": self.last_tire_change,
                "recent_trips": self.trip_history[-5:]
            },
            "diagnostics": {
                "oil": self.get_oil_status(),
                "tires": self.get_tires_status(),
                "engine": self.get_engine_analysis()
            }
        }
        return json.dumps(context, indent=4)

    def display_dashboard(self) -> None:
        oil_stat = self.get_oil_status()
        tire_stat = self.get_tires_status()
        engine_stat = self.get_engine_analysis()

        print("\n" + "="*50)
        print("  SYSTEM DIAGNOSTICS & LOGBOOK [PRO EDITION]")
        print("="*50)
        print(f" VEHICLE : {self.brand.upper()} {self.model.upper()}")
        print(f" MILEAGE : {self.mileage:,} (km) | TRIPS: {len(self.trip_history)}")
        print("-" * 50)
        print(f" OIL     : [{'' if 'CHANGE' in oil_stat else ''}] {oil_stat}")
        print(f" TIRES   : [{'' if 'CHECK' in tire_stat else ''}] {tire_stat}")
        print(f" ENGINE  :  {engine_stat['type']} ->  {engine_stat['tip']}")
        print("="*50)

    def save_data(self, silent: bool = False) -> None:
        data = {
            "brand": self.brand,
            "model": self.model,
            "engine_type": self.engine_type,
            "mileage": self.mileage,
            "last_oil_change": self.last_oil_change,
            "last_tire_change": self.last_tire_change,
            "trip_history": self.trip_history
        }
        with open("car_data.json", "w", encoding="utf-8") as file:
            json.dump(data, file, indent=4)
        if not silent:
            print(" [System] Data saved to disk.")

# --- Main Execution ---
def get_valid_integer(prompt: str, min_val: int = 0) -> int:
    """Helper function to cleanly handle integer inputs."""
    while True:
        try:
            val = int(input(prompt).strip())
            if val < min_val:
                print(f"   [Error] Value cannot be less than {min_val}.")
                continue
            return val
        except ValueError:
            print("   [Error] Invalid input. Numbers only.")

if __name__ == "__main__":
    clear_screen()
    print("== Smart Diagnostics Initialized ==\n")

    my_car = None

    if os.path.exists("car_data.json"):
        try:
            with open("car_data.json", "r", encoding="utf-8") as file:
                data = json.load(file)

            my_car = Car(
                brand=data.get("brand", "Unknown"),
                model=data.get("model", "Unknown"),
                engine_type=data.get("engine_type", "Unknown"),
                mileage=data.get("mileage", 0),
                last_oil_change=data.get("last_oil_change", 0),
                last_tire_change=data.get("last_tire_change", 0),
                trip_history=data.get("trip_history", [])
            )
            print(f"[System] Loaded: {my_car.brand} {my_car.model}\n")
        except json.JSONDecodeError:
            print(" [Error] Data file corrupted. Starting fresh.\n")

    if my_car is None:
        print("[*] Enter new vehicle details:")
        u_brand = input(" Brand  : ").strip()
        u_model = input(" Model  : ").strip()
        u_engine = input(" Engine : ").strip()
        u_mileage = get_valid_integer(" Mileage: ")

        my_car = Car(u_brand, u_model, u_engine, u_mileage)
        my_car.save_data(silent=True)

    # Interactive Loop
    while True:
        clear_screen()
        my_car.display_dashboard()

        print("\n[Menu Options]:")
        print(" 1. Add Trip         ")
        print(" 2. Reset Oil        ")
        print(" 3. Reset Tires      ")
        print(" 4. Edit Vehicle     ")
        print(" 5. View AI Context  ")
        print(" 6. Exit             ")

        choice = input("\nSelect (1-6): ").strip()

        if choice == '1':
            dist = get_valid_integer("\nTrip distance (KM): ", min_val=1)
            if my_car.add_trip(dist):
                input("\nPress Enter...")
        elif choice == '2':
            my_car.reset_service("oil")
            input("\nPress Enter...")
        elif choice == '3':
            my_car.reset_service("tires")
            input("\nPress Enter...")
        elif choice == '4':
            my_car.edit_data()
            input("\nPress Enter...")
        elif choice == '5':
            print("\n[AI JSON Context]:")
            print(my_car.get_llm_context())
            input("\nPress Enter...")
        elif choice == '6':
            print("\n" + "*"*40)
            print(" Safely powering down. Goodbye!")
            print("*"*40)
            input("\n Press Enter to close the terminal..")
            break


  SYSTEM DIAGNOSTICS & LOGBOOK [PRO EDITION]
 VEHICLE : M4 M4
 MILEAGE : 141,500 (km) | TRIPS: 1
--------------------------------------------------
 OIL     : [] CHANGE_REQUIRED (141,500 km driven)
 TIRES   : [] CHECK_TREAD (141,500 km driven)
 ENGINE  :  DIESEL/CRDi ->  Check Glow Plugs & EGR Valve

[Menu Options]:
 1. Add Trip         
 2. Reset Oil        
 3. Reset Tires      
 4. Edit Vehicle     
 5. View AI Context  
 6. Exit             

Select (1-6): 6

****************************************
 Safely powering down. Goodbye!
****************************************

 Press Enter to close the terminal..
